wind test yay

In [91]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import math

In [69]:
def load_df(file_name):
    """
    Loads the ERA5 data from a CSV file into a pandas DataFrame.
    """
    df = pd.read_csv(file_name, index_col=0, parse_dates=True)
    return df

In [70]:
df = load_df("data/era5_november_2026_data.csv")
df

C:\Users\james\AppData\Local\Temp\ipykernel_22392\107451435.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(file_name, index_col=0, parse_dates=True)


,valid_time,pressure_level,latitude,longitude,t,u,v,w,number,expver
id,,,,,,,,,,
0,2025-11-01,1000.0,43.0,-80.00,278.51270,2.199965,0.133451,0.026815,0,1
1,2025-11-01,1000.0,43.0,-79.75,278.68457,2.645766,0.173002,0.037679,0,1
2,2025-11-01,1000.0,43.0,-79.50,278.88574,3.543715,0.200834,0.046468,0,1
3,2025-11-01,1000.0,43.0,-79.25,278.93262,3.798110,0.277006,-0.003520,0,1
4,2025-11-01,1000.0,43.0,-79.00,278.81738,3.061294,0.398100,-0.063273,0,1
...,...,...,...,...,...,...,...,...,...,...
31852,2025-11-01,1.0,38.0,-71.00,253.41280,71.787094,8.202270,0.000045,0,1
31853,2025-11-01,1.0,38.0,-70.75,253.43819,72.001940,8.276489,0.000078,0,1
31854,2025-11-01,1.0,38.0,-70.50,253.46846,72.208970,8.352661,0.000109,0,1


In [71]:
target_lat, target_lon = 43, -79.625
target_pressure = 334

In [81]:
coords = np.array(df[['latitude', 'longitude']].values)
target = np.array([[target_lat, target_lon]])

In [82]:
coords

array([[ 43.  , -80.  ],
       [ 43.  , -79.75],
       [ 43.  , -79.5 ],
       ...,
       [ 38.  , -70.5 ],
       [ 38.  , -70.25],
       [ 38.  , -70.  ]], shape=(31857, 2))

In [83]:
target

array([[ 43.   , -79.625]])

In [84]:
distances = cdist(coords, target, metric='euclidean')
distances

array([[ 0.375     ],
       [ 0.125     ],
       [ 0.125     ],
       ...,
       [10.40507689],
       [10.625     ],
       [10.8462263 ]], shape=(31857, 1))

In [85]:
dist_flat = distances[:, 0]
dist_flat

array([ 0.375     ,  0.125     ,  0.125     , ..., 10.40507689,
       10.625     , 10.8462263 ], shape=(31857,))

In [76]:
min_dist = dist_flat.min()
min_dist

np.float64(0.125)

In [86]:
nearest_idx = np.where(np.isclose(dist_flat, min_dist, atol=1e-9))[0]
nearest_idx

array([    1,     2,   862,   863,  1723,  1724,  2584,  2585,  3445,
        3446,  4306,  4307,  5167,  5168,  6028,  6029,  6889,  6890,
        7750,  7751,  8611,  8612,  9472,  9473, 10333, 10334, 11194,
       11195, 12055, 12056, 12916, 12917, 13777, 13778, 14638, 14639,
       15499, 15500, 16360, 16361, 17221, 17222, 18082, 18083, 18943,
       18944, 19804, 19805, 20665, 20666, 21526, 21527, 22387, 22388,
       23248, 23249, 24109, 24110, 24970, 24971, 25831, 25832, 26692,
       26693, 27553, 27554, 28414, 28415, 29275, 29276, 30136, 30137,
       30997, 30998])

In [87]:
nearest_rows = df.iloc[nearest_idx].copy()
nearest_rows

,valid_time,pressure_level,latitude,longitude,t,u,v,w,number,expver
id,,,,,,,,,,
1,2025-11-01,1000.0,43.0,-79.75,278.68457,2.645766,0.173002,0.037679,0,1
2,2025-11-01,1000.0,43.0,-79.50,278.88574,3.543715,0.200834,0.046468,0,1
862,2025-11-01,975.0,43.0,-79.75,277.36450,4.851131,0.240036,0.117287,0,1
863,2025-11-01,975.0,43.0,-79.50,277.49340,5.467830,0.287399,0.105080,0,1
1723,2025-11-01,950.0,43.0,-79.75,275.86328,6.973121,0.084930,0.172542,0,1
...,...,...,...,...,...,...,...,...,...,...
29276,2025-11-01,3.0,43.0,-79.50,233.34908,26.954071,-3.738190,0.000222,0,1
30136,2025-11-01,2.0,43.0,-79.75,240.92891,36.830550,-1.852463,0.000177,0,1
30137,2025-11-01,2.0,43.0,-79.50,240.96211,37.045395,-1.793869,0.000176,0,1


In [88]:
pressure_idx = (nearest_rows['pressure_level'] - target_pressure).abs().idxmin()
pressure_idx

np.int64(15499)

In [89]:
final_row = nearest_rows.loc[pressure_idx]
final_row

valid_time        2025-11-01
pressure_level         350.0
latitude                43.0
longitude             -79.75
t                  233.73138
u                   30.29541
v                   -6.24387
w                   0.001613
number                     0
expver                     1
Name: 15499, dtype: object

In [90]:
def get_pressure(h):
    if h <= 11000:
        return 1013.25 * math.pow((1 - (0.0065 * h) / 288.15), 5.25588)
    elif h <= 20000:
        return 226.32 * math.pow(math.e, (-0.00015771 * (h - 11000)))
    else:
        return 54.749 * math.pow((216.65 / (216.65 + 0.001 * (h - 20000))), 34.163)

In [93]:
seq = np.arange(0, 30000, 1000)
for h in seq:
    print(get_pressure(h))

1013.25
898.7456253218028
794.9520124674402
701.0852557600716
616.4021264793947
540.1988697742273
471.81000900213326
410.6071576327686
355.997838857668
307.42431305567357
264.36241327015694
226.32038886166407
193.29933133374234
165.0964629465885
141.00846541685016
120.4349685289602
102.86319762216468
87.85519317433294
75.03689508128015
64.08881956774316
54.73809636096508
46.779000527112785
39.99805654947834
34.22450594074376
29.305086742524438
25.110397068206964
21.53109666942876
18.474726436779005
15.863043172798845
13.629784383172575
